# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*



One row is one webpage's search performance, for one client, on one single day, in March 2026 (2026-03-01 to 2026-03-31).

Table used: `fact_content_daily_performance`.

In [1]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
base = "hf://datasets/FlyRank/internship-warehouse"

print("Connected.")

Connected.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Label:** whether a page's average search position got worse from the first half of March (1st-15th) to the second half (16th-31st).

**Features (from the first half only):** average impressions, average clicks, click-through rate, average position, number of active days with impressions.

**Context (used for joining/splitting, not as predictors):** client ID, page ID, report date.

**Excluded:** rows where `gsc_data_available` is not TRUE. Only 36.7% of rows have search data available, and this is a client-level gap, not a timing issue (checked below).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Query 1 of 3 — grain**

In [3]:
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) as row_count
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()
print(f"Duplicate grain combos found: {len(grain_check)}")

total_rows = con.sql(f"""
    SELECT COUNT(*) as n
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(f"Total rows in month=2026-03: {total_rows['n'][0]}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain combos found: 0
Total rows in month=2026-03: 9841378


**Query 2 of 3 — row count and date span**

In [4]:
date_span = con.sql(f"""
    SELECT MIN(report_date) as earliest, MAX(report_date) as latest
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(date_span)

    earliest     latest
0 2026-03-01 2026-03-31


**Query 3 of 3 — availability, using IS TRUE**

In [5]:
availability = con.sql(f"""
    SELECT
        COUNT(*) as total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) as gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) as ga4_available_rows
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(availability)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  gsc_available_rows  ga4_available_rows
0     9841378             3611061              413966


**Five features (max), each knowable at the decision moment:**

- `avg_impressions_h1` — known once March 1-15 has passed
- `avg_clicks_h1` — known once March 1-15 has passed
- `ctr_h1` — derived from the two above, same window
- `avg_position_h1` — known from GSC data through March 15
- `active_days_h1` — known from the same window

In [6]:
page_level = con.sql(f"""
    WITH first_half AS (
        SELECT content_hash_id,
               AVG(gsc_impressions) as avg_impressions_h1,
               AVG(gsc_clicks) as avg_clicks_h1,
               SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) as ctr_h1,
               AVG(gsc_avg_position) as avg_position_h1,
               COUNT(*) FILTER (WHERE gsc_impressions > 0) as active_days_h1
        FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE AND report_date <= '2026-03-15'
        GROUP BY content_hash_id
    ),
    second_half AS (
        SELECT content_hash_id,
               SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) as ctr_h2,
               AVG(gsc_avg_position) as avg_position_h2
        FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE AND report_date > '2026-03-15'
        GROUP BY content_hash_id
    )
    SELECT f.*, s.ctr_h2, s.avg_position_h2
    FROM first_half f
    JOIN second_half s ON f.content_hash_id = s.content_hash_id
""").df()

print(page_level.shape)
page_level.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(141467, 8)


,content_hash_id,avg_impressions_h1,avg_clicks_h1,ctr_h1,avg_position_h1,active_days_h1,ctr_h2,avg_position_h2
0,content_b7e512995f79d5a6,28.600000,0.133333,0.004662,4.247255,15,0.000000,4.532026
1,content_05597932fe4da067,1.636364,0.000000,0.000000,4.939394,11,0.000000,1.083333
2,content_905aa32a0230694e,5.933333,0.000000,0.000000,3.010741,15,0.000000,9.952165
3,content_05434271b257bb68,41.866667,0.066667,0.001592,5.330069,15,0.006305,7.248714
4,content_d056587ff7faca0c,85.333333,0.600000,0.007031,4.468441,15,0.004698,4.450357


**The trap:** add the second-half outcome value as a feature on purpose, and watch the score jump toward perfect.

In [7]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

page_level["needs_refresh"] = (page_level["avg_position_h2"] > page_level["avg_position_h1"]).astype(int)
print(f"Label balance: {page_level['needs_refresh'].mean():.2%}")

honest_features = ["avg_impressions_h1", "avg_clicks_h1", "ctr_h1", "avg_position_h1", "active_days_h1"]

X = page_level[honest_features].fillna(0)
y = page_level["needs_refresh"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"Honest AUC (features from h1 only): {honest_auc:.4f}")

# Now add the label-derived column on purpose
leaky_features = honest_features + ["avg_position_h2"]
X_leaky = page_level[leaky_features].fillna(0)
X_train_leaky, X_test_leaky, y_train, y_test = train_test_split(
    X_leaky, y, test_size=0.25, random_state=42, stratify=y
)

leaky_model = LogisticRegression(max_iter=1000)
leaky_model.fit(X_train_leaky, y_train)
leaky_auc = roc_auc_score(y_test, leaky_model.predict_proba(X_test_leaky)[:, 1])

print(f"Honest AUC (h1 features only):        {honest_auc:.4f}")
print(f"Leaky AUC (h1 features + h2 position): {leaky_auc:.4f}")

# Delete the leaky feature, keep the honest number
final_features = honest_features
print(f"Final, honest AUC to report: {honest_auc:.4f}")

Label balance: 54.06%
Honest AUC (features from h1 only): 0.6366
Honest AUC (h1 features only):        0.6366
Leaky AUC (h1 features + h2 position): 1.0000
Final, honest AUC to report: 0.6366


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can only show association, not cause: it cannot say why a page's ranking changed. Only about a third of rows have real search data at all, and that gap is client-level (some clients never connected search console), not a timing artifact, since daily availability stayed steady across March. The client panel is also uneven, with some clients holding far more pages than others, so results may not generalize evenly across all clients.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.